# Training Loop Example

In this notebook, we implement a training loop for a basic transformer model in pyTorch on the tiny shakespeare data set, as a hommage to nanoGPT from Andrej Karpathy's brilliant [Neural Networks: Zero to Hero](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) series.

## Imports and Setup

In [84]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [85]:
from adam import Adam

In [86]:
file_path = "../data/input.txt"

with open(file_path, "r") as f:
    dataset = list(f.read())

## Dataset Preparation

In [87]:
chars = sorted(set(dataset))
ctoi = {c:i for i, c in enumerate(chars)}
itoc = {ctoi[c]:c for c in ctoi.keys()}

In [88]:
encode = lambda s: [ctoi[c] for c in s]
decode = lambda s: "".join([itoc[i] for i in s])

In [89]:
data = encode(dataset)
data_tensor = torch.tensor(data, dtype=torch.long)
data_size = data_tensor.size(dim=0)

In [90]:
split = round(data_size*0.9)
data_train = data_tensor[:split]
data_val = data_tensor[split:]

In [91]:
len_train = data_train.size(dim=0)
len_val = data_val.size(dim=0)

## Model Initialization

In [92]:
class Model(nn.Module):

    def __init__(self, block_size: int, vocab_size: int, d_model: int, nhead: int, num_layers: int):
        super().__init__()
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.positional_embedding = nn.Embedding(block_size, d_model)
        self.causal_mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()

        self.encoding_layer = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.encoding_layer, num_layers, enable_nested_tensor=False)

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, inputs: torch.Tensor):
        embeddings = self.token_embedding(inputs)
        pos_tensor = torch.tensor(range(self.block_size), dtype=torch.long)
        pos_embeddings = self.positional_embedding(pos_tensor)
        embeddings += pos_embeddings

        transformer_outputs = self.transformer(embeddings, self.causal_mask)
        logits = self.lm_head(transformer_outputs)
        return logits

In [93]:
block_size = 32
vocab_size = len(chars)
d_model = 15
nhead = 3
num_layers = 5

## Training Loop

In [94]:
model = Model(block_size, vocab_size, d_model, nhead, num_layers)

In [95]:
num_iters = 1000
params = model.parameters()

optim = Adam(params)

In [96]:
from torch import dtype


for _ in range(num_iters):
    sequence_idx = np.random.randint(0, len_train - block_size - 1)
    x = data_train[sequence_idx : sequence_idx + block_size]
    y = data_train[sequence_idx + 1 : sequence_idx + block_size + 1]
    
    logits = model(x)
    loss = F.cross_entropy(logits, y)

    optim.zero_grad()
    loss.backward()
    optim.step()

KeyError: 'params'